In [1]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib


In [2]:

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive')
    except ValueError as e:
        print(f"Error mounting Google Drive: {e}. Please ensure you authorize access when prompted.")
    data_path = '/content/drive/MyDrive/f1-telemetry-ml'
else:
    data_path = '../fastf1_data/'

labeled_path = f'{data_path}/labeled'
processed_path = f'{data_path}/processed'


In [3]:

target_cols = ['aggression_score', 'line_shape_score', 'oversteer_preference_score']

train_labels = pd.read_parquet(f'{labeled_path}/train_data.parquet')
val_labels = pd.read_parquet(f'{labeled_path}/val_data.parquet')
test_in_dist_labels = pd.read_parquet(f'{labeled_path}/test_in_dist_data.parquet')
test_zero_shot_labels = pd.read_parquet(f'{labeled_path}/test_zero_shot.parquet')

segmented_files = glob.glob(f'{processed_path}/corners_*.parquet')
segmented = pd.concat([pd.read_parquet(f) for f in segmented_files], ignore_index=True)

SAFE_COLUMNS = ['Throttle', 'Brake', 'Speed', 'nGear', 'RPM']
merge_keys = ['year', 'race', 'session_type', 'driver', 'lap_number', 'corner_number']

def build_flat_features(labels_df, segmented_df):
    merged = segmented_df.merge(labels_df[merge_keys + target_cols], on=merge_keys, how='inner')
    rows = []
    for keys, group in merged.groupby(merge_keys):
        row = dict(zip(merge_keys, keys))
        for col in SAFE_COLUMNS:
            row[f'{col}_mean'] = group[col].mean()
            row[f'{col}_std'] = group[col].std()
            row[f'{col}_min'] = group[col].min()
            row[f'{col}_max'] = group[col].max()
        for t in target_cols:
            row[t] = group[t].iloc[0]
        rows.append(row)
    return pd.DataFrame(rows)

print("Building flat features (this takes a few minutes on the full dataset)...")
train_df = build_flat_features(train_labels, segmented)
val_df = build_flat_features(val_labels, segmented)
test_in_dist_df = build_flat_features(test_in_dist_labels, segmented)
test_zero_shot_df = build_flat_features(test_zero_shot_labels, segmented)

feature_cols = [c for c in train_df.columns if c.endswith(('_mean', '_std', '_min', '_max'))]
print(f"Train: {train_df.shape}, Val: {val_df.shape}")

# --- Train ---
model = MultiOutputRegressor(Ridge(alpha=1.0))
model.fit(train_df[feature_cols], train_df[target_cols])


def evaluate_split(df, split_name):
    preds = model.predict(df[feature_cols])
    mae = mean_absolute_error(df[target_cols], preds)
    mse = mean_squared_error(df[target_cols], preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(df[target_cols], preds)
    print(f"\n--- Ridge — {split_name} ---")
    print(f"Overall MAE: {mae:.4f}  RMSE: {rmse:.4f}  R2: {r2:.4f}")
    for i, col in enumerate(target_cols):
        col_mae = mean_absolute_error(df[col], preds[:, i])
        col_mse = mean_squared_error(df[col], preds[:, i])
        col_rmse = np.sqrt(col_mse)
        col_r2 = r2_score(df[col], preds[:, i])
        print(f"  {col} — MAE: {col_mae:.4f}  RMSE: {col_rmse:.4f}  R2: {col_r2:.4f}")

evaluate_split(val_df, "Validation")
evaluate_split(test_in_dist_df, "Test (in-distribution)")
evaluate_split(test_zero_shot_df, "Test (zero-shot)")

Building flat features (this takes a few minutes on the full dataset)...
Train: (73848, 29), Val: (32350, 29)

--- Ridge — Validation ---
Overall MAE: 0.0575  RMSE: 0.0800  R2: 0.8399
  aggression_score — MAE: 0.0520  RMSE: 0.0769  R2: 0.7360
  line_shape_score — MAE: 0.0424  RMSE: 0.0552  R2: 0.9553
  oversteer_preference_score — MAE: 0.0783  RMSE: 0.1013  R2: 0.8286

--- Ridge — Test (in-distribution) ---
Overall MAE: 0.0611  RMSE: 0.0828  R2: 0.8450
  aggression_score — MAE: 0.0625  RMSE: 0.0870  R2: 0.6926
  line_shape_score — MAE: 0.0365  RMSE: 0.0465  R2: 0.9742
  oversteer_preference_score — MAE: 0.0844  RMSE: 0.1041  R2: 0.8683

--- Ridge — Test (zero-shot) ---
Overall MAE: 0.0651  RMSE: 0.0931  R2: 0.8773
  aggression_score — MAE: 0.0729  RMSE: 0.0969  R2: 0.8020
  line_shape_score — MAE: 0.0413  RMSE: 0.0518  R2: 0.9669
  oversteer_preference_score — MAE: 0.0811  RMSE: 0.1179  R2: 0.8630


In [4]:
from sklearn.model_selection import GridSearchCV, GroupKFold

# --- Hyperparameter tuning for Ridge ---

param_grid = {
    'estimator__alpha': [0.001, 0.01, 0.1, 1.0, 10.0, 50.0, 100.0, 200.0],
}

base_model = MultiOutputRegressor(Ridge())

# GroupKFold on driver — same principle as the original train/val split
group_kfold = GroupKFold(n_splits=5)
groups = train_df['driver']

search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=group_kfold,
    n_jobs=-1,
    verbose=2,
)

print("Running hyperparameter search for Ridge...")
search.fit(train_df[feature_cols], train_df[target_cols], groups=groups)

print(f"\nBest params: {search.best_params_}")
print(f"Best CV MAE: {-search.best_score_:.4f}")

# --- Evaluate the tuned model on all three splits, same as before ---
tuned_model = search.best_estimator_

def evaluate_split_tuned(df, split_name):
    preds = tuned_model.predict(df[feature_cols])
    mae = mean_absolute_error(df[target_cols], preds)
    rmse = np.sqrt(mean_squared_error(df[target_cols], preds))
    r2 = r2_score(df[target_cols], preds)
    print(f"\n--- Tuned Ridge — {split_name} ---")
    print(f"Overall MAE: {mae:.4f}  RMSE: {rmse:.4f}  R2: {r2:.4f}")
    for i, col in enumerate(target_cols):
        col_mae = mean_absolute_error(df[col], preds[:, i])
        col_rmse = np.sqrt(mean_squared_error(df[col], preds[:, i]))
        col_r2 = r2_score(df[col], preds[:, i])
        print(f"  {col} — MAE: {col_mae:.4f}  RMSE: {col_rmse:.4f}  R2: {col_r2:.4f}")

evaluate_split_tuned(val_df, "Validation")
evaluate_split_tuned(test_in_dist_df, "Test (in-distribution)")
evaluate_split_tuned(test_zero_shot_df, "Test (zero-shot)")

Running hyperparameter search for Ridge...
Fitting 5 folds for each of 8 candidates, totalling 40 fits

Best params: {'estimator__alpha': 1.0}
Best CV MAE: 0.0589

--- Tuned Ridge — Validation ---
Overall MAE: 0.0575  RMSE: 0.0800  R2: 0.8399
  aggression_score — MAE: 0.0520  RMSE: 0.0769  R2: 0.7360
  line_shape_score — MAE: 0.0424  RMSE: 0.0552  R2: 0.9553
  oversteer_preference_score — MAE: 0.0783  RMSE: 0.1013  R2: 0.8286

--- Tuned Ridge — Test (in-distribution) ---
Overall MAE: 0.0611  RMSE: 0.0828  R2: 0.8450
  aggression_score — MAE: 0.0625  RMSE: 0.0870  R2: 0.6926
  line_shape_score — MAE: 0.0365  RMSE: 0.0465  R2: 0.9742
  oversteer_preference_score — MAE: 0.0844  RMSE: 0.1041  R2: 0.8683

--- Tuned Ridge — Test (zero-shot) ---
Overall MAE: 0.0651  RMSE: 0.0931  R2: 0.8773
  aggression_score — MAE: 0.0729  RMSE: 0.0969  R2: 0.8020
  line_shape_score — MAE: 0.0413  RMSE: 0.0518  R2: 0.9669
  oversteer_preference_score — MAE: 0.0811  RMSE: 0.1179  R2: 0.8630


In [5]:
if IN_COLAB:
    os.makedirs(f'{data_path}/models', exist_ok=True)
    joblib.dump(model, f'{data_path}/models/ridge_model.pkl')
    joblib.dump(tuned_model, f'{data_path}/models/ridge_model_tuned.pkl')
else:
    os.makedirs(f'../models', exist_ok=True)
    joblib.dump(model, f'../models/ridge_model.pkl')
    joblib.dump(tuned_model, f'../models/ridge_model_tuned.pkl')
print("\nModels saved.")


Models saved.
